# European Streetview CNN Classifier — Speed-Optimised
Trains EfficientNet-B0 to classify 34 k streetview images by European country.

**Speed improvements vs previous version**

| What changed | Why it helps |
|---|---|
| `BATCH_SIZE 16 → 64` | Fewer kernel launches, better GPU utilisation |
| `persistent_workers=True` | Workers stay alive between epochs (saves ~10 s/epoch) |
| `prefetch_factor=4` | More batches queued ahead, GPU rarely waits |
| `cudnn.benchmark=True` | Picks fastest conv kernel for your exact input shape |
| `non_blocking=True` transfers | Overlaps CPU→GPU copy with compute |
| `set_to_none=True` in zero_grad | Frees memory faster, reduces fragmentation |
| `OneCycleLR` scheduler | Converges in fewer epochs than Cosine |
| 2-phase freeze/unfreeze | Head warms up first → stable gradients before touching backbone |
| `torch.compile()` (PyTorch ≥ 2) | ~10–30 % throughput boost via kernel fusion |
| 2× batch at inference | Faster eval / test loops |


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ── 1. Imports & reproducibility ────────────────────────────────────────────
import os, random, json
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = False   # ⚡ allow non-deterministic ops (faster)
torch.backends.cudnn.benchmark    = True     # ⚡ auto-tune conv kernels for fixed input size

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Using device: cpu


In [4]:
# ── 2. Configuration ─────────────────────────────────────────────────────────
TRAIN_DIR = '/content/drive/MyDrive/GeoGussrCheat/trainEurope'
TEST_DIR  = '/content/drive/MyDrive/GeoGussrCheat/testEurope'

IMG_SIZE   = 224
# ⚡ Increase batch size as large as your VRAM allows.
#    T4 (15 GB): 64–96. A100 (40 GB): 128–256. Adjust down if OOM.
BATCH_SIZE = 64

NUM_EPOCHS     = 30
FREEZE_EPOCHS  = 5      # ⚡ Freeze backbone for first N epochs — converges faster
LR_HEAD        = 3e-4   # ⚡ Higher LR for the head-only warm-up phase
LR_FULL        = 5e-5   # LR once backbone is unfrozen
VAL_SPLIT      = 0.25

# ⚡ Set to os.cpu_count() // 2 on Colab (usually 2); up to 8 on A100 instances.
NUM_WORKERS = min(4, os.cpu_count() or 2)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print(f'BATCH_SIZE={BATCH_SIZE}, NUM_WORKERS={NUM_WORKERS}, FREEZE_EPOCHS={FREEZE_EPOCHS}')


In [5]:
# ── 3. Dataset definition ────────────────────────────────────────────────────
class StreetviewDataset(Dataset):
    """Loads images from root/<country>/<img>.jpg"""

    def __init__(self, root_dir, transform=None, class_to_idx=None):
        self.root_dir  = root_dir
        self.transform = transform
        self.samples   = []   # list of (path, label_idx)

        countries = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])

        if class_to_idx is None:
            self.class_to_idx = {c: i for i, c in enumerate(countries)}
        else:
            self.class_to_idx = class_to_idx

        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        self.classes = countries

        IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
        for country in countries:
            if country not in self.class_to_idx:
                continue
            label  = self.class_to_idx[country]
            folder = os.path.join(root_dir, country)
            for fname in os.listdir(folder):
                if os.path.splitext(fname)[1].lower() in IMG_EXTS:
                    self.samples.append((os.path.join(folder, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


def stratified_split(dataset, val_fraction=0.25, seed=42):
    labels   = [s[1] for s in dataset.samples]
    indices  = list(range(len(labels)))
    train_idx, val_idx = train_test_split(
        indices, test_size=val_fraction, stratify=labels, random_state=seed
    )
    return train_idx, val_idx


full_train_ds = StreetviewDataset(TRAIN_DIR, transform=None)
NUM_CLASSES   = len(full_train_ds.class_to_idx)
print(f'Countries found: {NUM_CLASSES}')
print(f'Total training images: {len(full_train_ds)}')
print('Classes:', list(full_train_ds.class_to_idx.keys())[:10], '...')


Countries found: 37
Total training images: 32686
Classes: ['Albania', 'Andorra', 'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Czech Republic', 'Denmark', 'Estonia', 'Finland'] ...


In [6]:
# ── 4. Build train / val splits ──────────────────────────────────────────────
class TransformSubset(Dataset):
    """Wraps a Subset and applies a transform at access time."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


train_idx, val_idx = stratified_split(full_train_ds, val_fraction=VAL_SPLIT, seed=SEED)

train_subset = Subset(full_train_ds, train_idx)
val_subset   = Subset(full_train_ds, val_idx)

train_ds = TransformSubset(train_subset, train_tf)
val_ds   = TransformSubset(val_subset,   val_tf)

print(f'Train samples : {len(train_ds)}')
print(f'Val   samples : {len(val_ds)}')

_loader_kwargs = dict(
    num_workers      = NUM_WORKERS,
    pin_memory       = DEVICE.type == 'cuda',
    persistent_workers = NUM_WORKERS > 0,   # ⚡ keep workers alive between epochs
    prefetch_factor  = 4 if NUM_WORKERS > 0 else None,   # ⚡ buffer more batches ahead
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  **_loader_kwargs)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **_loader_kwargs)

steps_per_epoch = len(train_loader)
print(f'Steps per epoch: {steps_per_epoch}  (batch={BATCH_SIZE})')


Train samples : 24514
Val   samples : 8172


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [7]:
# ── 5. Model (EfficientNet-B0 fine-tuned) ───────────────────────────────────
def build_model(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes),
    )
    return model


model = build_model(NUM_CLASSES).to(DEVICE)

# ⚡ Freeze backbone for the first FREEZE_EPOCHS epochs (head-only warm-up).
#    This lets the new classification head stabilise before touching pretrained weights.
def set_backbone_grad(model, requires_grad: bool):
    for param in model.features.parameters():
        param.requires_grad = requires_grad

set_backbone_grad(model, requires_grad=False)   # start frozen
print('Backbone frozen for warm-up phase.')

# ⚡ torch.compile (PyTorch ≥ 2.0 on Colab) — ~10-30% throughput boost.
#    Comment out if you see errors on older runtimes.
try:
    model = torch.compile(model)
    print('torch.compile() applied.')
except Exception as e:
    print(f'torch.compile skipped: {e}')

total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}  |  Trainable (head): {trainable:,}')


Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=1536, out_features=512, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.3, inplace=False)
  (4): Linear(in_features=512, out_features=37, bias=True)
)
Trainable parameters: 11,502,157


In [8]:
# ── 6. Loss, optimiser, scheduler ───────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ⚡ Phase 1: head-only warm-up — only train classifier params
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=1e-4
)

# ⚡ OneCycleLR converges significantly faster than CosineAnnealingLR.
#    It ramps LR up then down within each epoch, functioning as a built-in warm-up.
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr      = LR_HEAD,
    steps_per_epoch = steps_per_epoch,
    epochs      = FREEZE_EPOCHS,       # will be recreated after unfreeze
    pct_start   = 0.3,
)
print(f'OneCycleLR phase-1: max_lr={LR_HEAD}, epochs={FREEZE_EPOCHS}')


In [ ]:
# ── 7. Training loop ─────────────────────────────────────────────────────────
# ⚡ Mixed-precision scaler
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')

GRAD_ACCUM = 1   # ⚡ Increase (e.g. 2–4) if you must use a small BATCH_SIZE
                 #    Effective batch = BATCH_SIZE * GRAD_ACCUM

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
_phase2_started = False


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        optimizer.zero_grad(set_to_none=True)   # ⚡ set_to_none frees memory faster
        for step, (imgs, labels) in enumerate(loader):
            # ⚡ non_blocking=True overlaps H→D transfer with compute
            imgs   = imgs.to(DEVICE,   non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                outputs = model(imgs)
                loss    = criterion(outputs, labels)
                if GRAD_ACCUM > 1:
                    loss = loss / GRAD_ACCUM

            if training:
                scaler.scale(loss).backward()
                if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()
            else:
                pass   # no zero_grad needed during validation

            total_loss += loss.item() * imgs.size(0) * (GRAD_ACCUM if training and GRAD_ACCUM > 1 else 1)
            preds    = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += imgs.size(0)

    return total_loss / total, correct / total


print(f"{'Epoch':>6} | {'Phase':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7} | {'LR':>9}")
print('-' * 75)

for epoch in range(1, NUM_EPOCHS + 1):

    # ⚡ Phase 2: unfreeze backbone after FREEZE_EPOCHS, reset optimizer + scheduler
    if epoch == FREEZE_EPOCHS + 1 and not _phase2_started:
        _phase2_started = True
        set_backbone_grad(model, requires_grad=True)
        optimizer = optim.AdamW(model.parameters(), lr=LR_FULL, weight_decay=1e-4)
        remaining = NUM_EPOCHS - FREEZE_EPOCHS
        scheduler = optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=LR_FULL,
            steps_per_epoch=steps_per_epoch,
            epochs=remaining, pct_start=0.1,
        )
        print(f'  → Backbone unfrozen. Phase-2 OneCycleLR: max_lr={LR_FULL}, epochs={remaining}')

    phase = 'freeze' if epoch <= FREEZE_EPOCHS else 'full'
    tr_loss, tr_acc = run_epoch(train_loader, training=True)
    vl_loss, vl_acc = run_epoch(val_loader,   training=False)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        # ⚡ Save underlying module if model is compiled
        state = model._orig_mod.state_dict() if hasattr(model, '_orig_mod') else model.state_dict()
        torch.save(state, 'best_model.pth')

    lr_now = optimizer.param_groups[0]['lr']
    print(f'{epoch:>6} | {phase:>6} | {tr_loss:>10.4f} | {tr_acc:>8.2%} | {vl_loss:>8.4f} | {vl_acc:>6.2%} | {lr_now:>9.2e}')

print(f'\nBest validation accuracy: {best_val_acc:.2%}')


 Epoch | Train Loss | Train Acc | Val Loss | Val Acc
-------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# ── 8. Loss & accuracy curves ────────────────────────────────────────────────
epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs, history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   linewidth=2, linestyle='--')
axes[0].set_title('Loss Curve', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, [a * 100 for a in history['train_acc']], label='Train Acc', linewidth=2)
axes[1].plot(epochs, [a * 100 for a in history['val_acc']],   label='Val Acc',   linewidth=2, linestyle='--')
axes[1].set_title('Accuracy Curve', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('EfficientNet-B3 – European Streetview Classification', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Curves saved to training_curves.png')

In [ ]:
# ── 9. Evaluate on testEurope ────────────────────────────────────────────────
model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
model.eval()

test_ds = StreetviewDataset(
    TEST_DIR,
    transform=val_tf,
    class_to_idx=full_train_ds.class_to_idx
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE * 2, shuffle=False,   # ⚡ 2× batch at inference
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == 'cuda',
    persistent_workers=NUM_WORKERS > 0,
)

print(f'Test images: {len(test_ds)}')

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):   # ⚡ AMP at inference too
            outputs = model(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

test_acc = np.mean(np.array(all_preds) == np.array(all_labels))
print(f'\nTest Accuracy: {test_acc:.2%}')


In [ ]:
# ── 10. Per-country classification report ───────────────────────────────────
class_names = [full_train_ds.idx_to_class[i] for i in range(NUM_CLASSES)]

print(classification_report(
    all_labels, all_preds,
    target_names=class_names,
    digits=3
))

In [ ]:
# ── 11. Confusion matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

fig_h = max(10, NUM_CLASSES // 2)
plt.figure(figsize=(fig_h + 2, fig_h))
sns.heatmap(
    cm, annot=(NUM_CLASSES <= 20),
    fmt='d', cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title('Confusion Matrix – testEurope', fontsize=14)
plt.ylabel('True Country')
plt.xlabel('Predicted Country')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved to confusion_matrix.png')

In [ ]:
# ── 12. Quick inference helper ───────────────────────────────────────────────
def predict_image(image_path, model, class_to_idx, transform, device, top_k=5):
    """Return top-k predicted countries for a single image."""
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0]

    top_probs, top_indices = torch.topk(probs, k=top_k)
    results = [(idx_to_class[i.item()], p.item()) for i, p in zip(top_indices, top_probs)]

    # Display
    fig, axes = plt.subplots(1, 2, figsize=(10, 4),
                              gridspec_kw={'width_ratios': [1, 2]})
    axes[0].imshow(img)
    axes[0].axis('off')
    axes[0].set_title('Input Image')

    countries = [r[0] for r in results]
    confidences = [r[1] * 100 for r in results]
    bars = axes[1].barh(countries[::-1], confidences[::-1], color='steelblue')
    axes[1].set_xlabel('Confidence (%)')
    axes[1].set_title(f'Top-{top_k} Predictions')
    axes[1].set_xlim(0, 100)
    for bar, conf in zip(bars, confidences[::-1]):
        axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                     f'{conf:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
    return results


# Example usage – replace with any image path:
# predict_image('testEurope/France/img_001.jpg', model, full_train_ds.class_to_idx, val_tf, DEVICE)